### 1. Calculo de metricas
Para este notebook se requiere utilizar hasta Python 3.10, para que la libreria AligScore se deben contar con versiones especificas que pueden hacer funcionar mal los procesos de fine tuning por lo que se dejan por separado, la idea es tomar todos los archivos csv que cuentan con el texto cientifico, texto resumen original y el resumen generado por medio del LLM en este notebook y realizar el calculo de las metricas: Legibilidad, Relevancia y Factualidad.

In [23]:
!pip install --quiet  -r req-fine-models-metrics.txt

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [22]:
import pandas as pd
import numpy as np
import textstat
from typing import List, Dict, Any, Optional, Tuple
from bert_score import score as bert_score
import torch
from pathlib import Path
DATA_ALIGN = Path("./models/alignscore")
DATA_ALIGN.mkdir(parents=True, exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cpu


In [ ]:
### Modelo requerido base, puede utilizarse large tambien.

!(cd models/alignscore; curl -O -OL https://huggingface.co/yzha/AlignScore/resolve/main/AlignScore-base.ckpt)


  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


100  1321  100  1321    0     0     17      0  0:01:17  0:01:15  0:00:02   339-:--     0-     0
100 1875M  100 1875M    0     0  18.3M      0  0:01:42  0:01:42 --:--:-- 69.4M1k      0  0:27:33  0:01:16  0:26:17 77.2M   0  0:01:57  0:01:37  0:00:20 65.3M


In [15]:
def evaluar_factualidad_alignscore(preds, refs,evaluation_mode, batch_size, device,flag_threshold: float = 0.5):
#evaluation_mode,    # 'nli_sp' (por defecto AlignScore), 'nli', 'bin_sp', 'bin'
    assert len(preds) == len(refs), "preds y refs deben tener la misma longitud"

    # Import tardío para que esta función siga importando aunque no esté instalada la lib.
    from alignscore import AlignScore  
    # Inicializar scorer
    backbone = 'roberta-base'
    scorer = AlignScore(
        model="roberta-base",
        batch_size=batch_size,
        device=device,
        ckpt_path='models/alignscore/AlignScore-base.ckpt',
        evaluation_mode=evaluation_mode
    )

    scores = scorer.score(contexts=refs, claims=preds) 
    scores = [float(s) for s in scores]

    flags_low = [bool(s < flag_threshold) for s in scores]
    per_example = pd.DataFrame({"alignscore": scores,"flag_low": flags_low}) 

    summary = {
        "mean_alignscore": float(np.mean(scores)) if scores else float("nan"),
        "std_alignscore":  float(np.std(scores)) if scores else float("nan"),
        "min_alignscore":  float(np.min(scores)) if scores else float("nan"),
        "max_alignscore":  float(np.max(scores)) if scores else float("nan"),
        "n_examples":      int(len(scores)),
        "backbone":        backbone,
        "evaluation_mode": evaluation_mode,
        "batch_size":      int(batch_size),
        "device":          device,
        "ckpt_path":       'models/alignscore/AlignScore-base.ckpt',
        "flag_threshold":  float(flag_threshold)
    }

    return summary, per_example



In [8]:
def calcular_bertscore_relevancia(preds,refs,idf,rescale_with_baseline,batch_size,device):

    modelo = "roberta-large"
    lang = 'en'
    assert len(preds) == len(refs), "preds y refs deben tener la misma longitud"


    P, R, F1 = bert_score(
        cands=preds,
        refs=refs,
        lang=lang,
        model_type=modelo,
        idf=idf,
        rescale_with_baseline=rescale_with_baseline,
        batch_size=batch_size,
        device=device
    )

    p_list = [float(p) for p in P]
    r_list = [float(r) for r in R]
    f1_list = [float(f) for f in F1]

    summary = {
        "mean_precision": float(np.mean(p_list)) if p_list else float("nan"),
        "mean_recall":    float(np.mean(r_list)) if r_list else float("nan"),
        "mean_f1":        float(np.mean(f1_list)) if f1_list else float("nan"),
        "backbone_for_bertscore": modelo,
        "idf": bool(idf),
        "rescale_with_baseline": bool(rescale_with_baseline),
        "batch_size": int(batch_size),
        "device": device if device is not None else "auto"
    }

    per_example = {
        "bertscore_precision": p_list,
        "bertscore_recall": r_list,
        "bertscore_f1": f1_list
    }

    per_example = pd.DataFrame(per_example)

    return summary, per_example


In [ ]:

def calcular_legibilidad_textstat(preds):
    lang = 'en'
    textstat.set_lang(lang)
    rows = []
    for t in preds:
        t = t or ""

        row = {
            "flesch_reading_ease":  float(textstat.flesch_reading_ease(t)),
            "flesch_kincaid_grade": float(textstat.flesch_kincaid_grade(t)),
        }
        row.update({
            "gunning_fog":              float(textstat.gunning_fog(t)),
            "smog_index":               float(textstat.smog_index(t)) if textstat.sentence_count(t) >= 3 else float("nan"),
            "dale_chall":               float(textstat.dale_chall_readability_score(t)),
            "automated_readability":    float(textstat.automated_readability_index(t)),
            "coleman_liau":             float(textstat.coleman_liau_index(t)),
            "text_standard":            textstat.text_standard(t, float_output=True),
            "num_sentences":            int(textstat.sentence_count(t)),
            "num_words":                int(textstat.lexicon_count(t, removepunct=True)),
            "syllables":                int(textstat.syllable_count(t)),
            "reading_time_sec":         float(textstat.reading_time(t)),
        })

        rows.append(row)

    def _try_mean(key: str):
        vals = [r[key] for r in rows if key in r and isinstance(r[key], (int, float)) and not np.isnan(r[key])]
        return float(np.mean(vals)) if vals else float("nan")

    keys = sorted({k for r in rows for k in r.keys()})
    summary = {"n_examples": len(preds), "lang": lang}
    for k in keys:
        summary[f"mean_{k}"] = _try_mean(k)

    per_example = pd.DataFrame(rows) 
    return summary, per_example


### Calculo de ejemplo de las 3 metricas requeridas

In [17]:
summary, per_example = calcular_bertscore_relevancia(
    ['PLS Generated summary'],['English original cientific text'],
    idf=False,#Set pequeno a false, sino dejar en TRUE
    rescale_with_baseline=False,
    batch_size=1,
    device=device
)

print(summary)     
per_example.head(1)

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


{'mean_precision': 0.828174352645874, 'mean_recall': 0.8150588870048523, 'mean_f1': 0.821564257144928, 'backbone_for_bertscore': 'roberta-large', 'idf': False, 'rescale_with_baseline': False, 'batch_size': 1, 'device': device(type='cpu')}


,bertscore_precision,bertscore_recall,bertscore_f1
0,0.828174,0.815059,0.821564


In [13]:
summary, per_example = calcular_legibilidad_textstat(['PLS Generated summary'])
print(summary)     
per_example.head(1)

{'n_examples': 1, 'lang': 'en', 'mean_automated_readability': 9.899999999999999, 'mean_coleman_liau': 11.066666666666666, 'mean_dale_chall': 19.5753, 'mean_flesch_kincaid_grade': 17.04666666666667, 'mean_flesch_reading_ease': -21.809999999999945, 'mean_gunning_fog': 27.86666666666667, 'mean_num_sentences': 1.0, 'mean_num_words': 3.0, 'mean_reading_time_sec': 0.27911, 'mean_smog_index': nan, 'mean_syllables': 8.0, 'mean_text_standard': 11.0}


,flesch_reading_ease,flesch_kincaid_grade,gunning_fog,smog_index,dale_chall,automated_readability,coleman_liau,text_standard,num_sentences,num_words,syllables,reading_time_sec
0,-21.81,17.046667,27.866667,NaN,19.5753,9.9,11.066667,11.0,1,3,8,0.27911


In [24]:
summary, per_example = evaluar_factualidad_alignscore(
    ['PLS Generated summary'], ['English original cientific text'],
    'nli_sp', 16, device)
print(summary)     
per_example.head(1)

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Lightning automatically upgraded your loaded checkpoint from v1.7.7 to v1.9.5. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint --file models/alignscore/AlignScore-base.ckpt`
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/Users/htorre/Documents/anaconda/anaconda3/envs/P310/lib/python3.10/site-packages/pytorch_lightning/core/saving.py:255: UserWarning: Found keys that are not in the model state dict b

{'mean_alignscore': 0.1338200867176056, 'std_alignscore': 0.0, 'min_alignscore': 0.1338200867176056, 'max_alignscore': 0.1338200867176056, 'n_examples': 1, 'backbone': 'roberta-base', 'evaluation_mode': 'nli_sp', 'batch_size': 16, 'device': device(type='cpu'), 'ckpt_path': 'models/alignscore/AlignScore-base.ckpt', 'flag_threshold': 0.5}


,alignscore,flag_low
0,0.13382,True
